In [ ]:
from dotenv import load_dotenv

load_dotenv("/home/ubuntu/work/edu-src-all/.env")
#load_dotenv()

In [ ]:
#!pip install openai
#!pip install langchain-core==0.2.17
#!pip install langchain-openai==0.1.17
#!pip install pydantic==1.10.8
#!pip install pydantic_core==2.18.2
!pip install -U openai langchain langchain-openai langchain-community
#!pip install openai langchain==0.3 langchain-openai==0.3 langchain-community==0.3


In [ ]:
!pip install google-search-results
!pip install wikipedia
!pip install faiss-cpu 
!pip install sentence_transformers 
!pip install tiktoken 

In [ ]:
# from langchain_core.pydantic_v1 import BaseModel, Field # 예전 방식

# from pydantic import BaseModel # 새로운 방식

from pydantic.v1 import BaseModel # pydantic.v1 호환이 필요한 경우

In [ ]:
# 예전에 사용한대로 AI, Human, System message를 배열로 전달

#from langchain.chat_models import ChatOpenAI
from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage
)

chat = ChatOpenAI(model_name='gpt-4o-mini', temperature=0.9)
sys = SystemMessage(content="당신은 음악 추천을 해주는 전문 AI입니다.")
msg = HumanMessage(content='1990년대 재즈 5곡만 추천해줘.')

aimsg = chat.invoke([sys, msg])
aimsg.content

In [ ]:
# 예전에 사용한대로 AI, Human, System message를 배열로 전달

from langchain_openai import ChatOpenAI
from langchain_core.messages import (

    AIMessage,
    HumanMessage,
    SystemMessage
)

chat = ChatOpenAI(model_name='gpt-4o-mini', temperature=0.9)
sys = SystemMessage(content="당신은 여행을 추천해주는 전문 AI입니다.")
msg1 = HumanMessage(content='베트남 사파에 1박 2일 여행 가면 충분할까?')
msg2 = AIMessage(content='사파는 1박 2일로 가기에는 너무 멉니다. 일정을 조정하거나 다른 여행지를 선택하시면 어떨까요?')
msg3 = HumanMessage(content='음 ... 그러면 어디가 좋을까?')

aimsg = chat.invoke([sys, msg1, msg2, msg3])
aimsg.content

In [ ]:
# PromptTemplate를 사용하여 각 LLM에 맞는 prompt를 지정할 수 있음

from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["상품"],
    template="{상품} 만드는 회사 이름 추천해줘. 기억에 남는 한글 이름으로",
)

prompt.format(상품="AI 여행 추천 서비스") # 변수가 여러 개 일 때, 지정

In [ ]:
chat = ChatOpenAI(model_name='gpt-4o-mini', temperature=0.9)
chain = prompt | chat

chain.invoke({"상품": "학원 추천 서비스"})

In [ ]:
# 각 Prompt 별 template 지정도 가능

from langchain_openai import ChatOpenAI
from langchain_core.prompts.chat import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)

llm = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)  

template="You are a helpful assisstant that translates {input_language} to {output_language}."
system_message_prompt = SystemMessagePromptTemplate.from_template(template)
human_template="{text}"
human_message_prompt = HumanMessagePromptTemplate.from_template(human_template)

chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt, human_message_prompt])

chatchain = chat_prompt | llm
chatchain.invoke({"input_language": "English", "output_language": "French", "text": "I love programming."})

In [ ]:
!pip install numexpr

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_community.tools import WikipediaQueryRun, Tool
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.tools import tool

llm = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)

wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

tools = [wikipedia, multiply]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant. Answer questions accurately in Korean.",
    debug=True
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "아마존의 창업자는 누구인가? 그의 나이와 배우자의 나이를 곱하면 얼마인가?"
        }
    ]
})

In [ ]:
result['messages'][-1].content

In [ ]:
agent

In [2]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationBufferMemory

llm = ChatOpenAI(model_name='gpt-4o-mini', temperature=0.9)

memory = ConversationBufferMemory()
conversation = ConversationChain(llm=llm, memory=memory, verbose=True)
conversation.predict(input="인공지능에서 LLM이 뭐야?")

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
conversation.predict(input="PLM과의 차이를 설명해줘.")

In [ ]:
conversation.memory

In [ ]:
# https://python.langchain.com/api_reference/community/document_loaders.html#module-langchain_community.document_loaders

from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(web_path="https://ko.wikipedia.org/wiki/IU")
documents = loader.load()
documents

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
docs = text_splitter.split_documents(documents)
len(docs)

In [ ]:
docs[:3]

In [ ]:
!pip install langchain-huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()
#embeddings = HuggingFaceEmbeddings() # sentence-transformers/all-mpnet-base-v2

from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(docs, embedding=embeddings)
vectorstore.save_local("faiss-nj-IU")
# db = FAISS.load_local("faiss-nj-IU", embeddings=embeddings, allow_dangerous_deserialization=True)

In [ ]:
index = vectorstore.as_retriever()

In [ ]:
index.invoke("아이유의 데뷔곡은?", llm=chat, verbose=True)

In [ ]:
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n".join([d.page_content for d in docs])

rag_prompt = ChatPromptTemplate.from_template("""
다음의 문맥을 이용하여 질문에 답변해주세요.

문맥:
{context}

질문: {question}
""")

rag_chain = (
    {"context": index | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | chat
)

In [ ]:
result = rag_chain.invoke("아이유의 나이는?")
result.content

In [ ]:
result = rag_chain.invoke("아이유의 나이는? (오늘은 2025년 11월 23일)")
result.content

In [ ]:
result = rag_chain.invoke("아이유는 지금까지 총 콘서트를 몇회했는가?")
result.content